# Basic sequence-only classifier: globular-like vs disordered-like proteins



We will build a very simple machine-learning workflow:

1. Retrieve **disordered proteins** from **DisProt**
2. Retrieve a set of **reviewed enzyme proteins** from **UniProt** as a rough proxy for **globular proteins**
3. Convert sequences into **simple numeric features**
4. Train a **logistic regression** model
5. Evaluate the model

## Important biological note

This is a **simplified teaching exercise**.  
We are **not** building a production-quality globularity predictor.

Instead, we use:

- **DisProt sequences** as examples of **disordered / non-globular-like** proteins
- **Reviewed UniProt enzyme sequences** as examples of **globular-like** proteins

So the model is really learning a **toy classification problem from sequence alone**.

## 0. Install / import packages

If `requests` or `scikit-learn` are missing, uncomment the install line.

In [ ]:

# Uncomment if needed:
# !pip install requests pandas numpy scikit-learn matplotlib

import io
import math
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    auc,
)

## 1. Configuration

These values keep the dataset small enough for a student practical.

In [ ]:

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

N_DISPROT = 200   # class 0: disordered / non-globular-like
N_UNIPROT = 200   # class 1: globular-like

MIN_LEN = 50
MAX_LEN = 500

DISPROT_URL = "https://disprot.org/api/search"
UNIPROT_URL = "https://rest.uniprot.org/uniprotkb/search"

## 2. Helper functions for downloading data

### DisProt
We query the DisProt API and extract sequences from returned entries.

### UniProt
We query the UniProt REST API and ask for tab-separated output with:
- accession
- protein name
- sequence
- length

For UniProt, we use a deliberately simple query:

- reviewed entries
- non-fragment proteins
- proteins with an EC number
- within a moderate length range

This gives us a rough set of **enzyme-like, folded proteins** for the teaching exercise.

In [ ]:

def fetch_disprot_sequences(max_n=200, min_len=50, max_len=500, timeout=60):
    """
    Fetch a small set of DisProt sequences through the DisProt REST API.

    Returns a DataFrame with columns:
        source_id, name, sequence, label
    label = 0 for disordered / non-globular-like
    """
    params = {
        "size": max_n * 3,   # oversample a bit before length filtering
    }

    r = requests.get(DISPROT_URL, params=params, timeout=timeout)
    r.raise_for_status()
    payload = r.json()

    # The exact JSON layout may evolve, so we try a few common shapes.
    if isinstance(payload, dict):
        entries = (
            payload.get("data")
            or payload.get("results")
            or payload.get("items")
            or payload.get("content")
            or []
        )
    elif isinstance(payload, list):
        entries = payload
    else:
        entries = []

    rows = []
    for entry in entries:
        seq = (
            entry.get("sequence")
            or entry.get("full_sequence")
            or entry.get("seq")
            or ""
        )
        if not isinstance(seq, str):
            continue
        seq = seq.strip().upper()
        if not (min_len <= len(seq) <= max_len):
            continue

        source_id = entry.get("disprot_id") or entry.get("id") or entry.get("acc") or "unknown"
        name = entry.get("name") or entry.get("protein_name") or entry.get("title") or "DisProt entry"

        rows.append({
            "source_id": source_id,
            "name": name,
            "sequence": seq,
            "label": 0,
        })

        if len(rows) >= max_n:
            break

    return pd.DataFrame(rows)


def fetch_uniprot_sequences(max_n=200, min_len=50, max_len=500, timeout=60):
    """
    Fetch a small set of reviewed UniProt proteins with EC numbers as a rough
    proxy for globular proteins.

    Returns a DataFrame with columns:
        source_id, name, sequence, label
    label = 1 for globular-like
    """
    # Keep the query simple and robust for teaching purposes.
    query = f"reviewed:true AND fragment:false AND existence:1 AND ec:* AND length:[{min_len} TO {max_len}]"

    params = {
        "query": query,
        "format": "tsv",
        "fields": "accession,protein_name,length,sequence",
        "size": max_n,
    }

    r = requests.get(UNIPROT_URL, params=params, timeout=timeout)
    r.raise_for_status()

    df = pd.read_csv(io.StringIO(r.text), sep="\t")
    df = df.rename(columns={
        "Entry": "source_id",
        "Protein names": "name",
        "Sequence": "sequence",
    })

    df["label"] = 1
    return df[["source_id", "name", "sequence", "label"]]

## 3. Download the two classes

If one of the APIs is temporarily unavailable, you can rerun this cell later.

In [ ]:

disprot_df = fetch_disprot_sequences(max_n=N_DISPROT, min_len=MIN_LEN, max_len=MAX_LEN)
uniprot_df = fetch_uniprot_sequences(max_n=N_UNIPROT, min_len=MIN_LEN, max_len=MAX_LEN)

print("DisProt examples:", len(disprot_df))
print("UniProt examples:", len(uniprot_df))

display(disprot_df.head())
display(uniprot_df.head())

## 4. Combine into one input table

This is the main input table for the practical.

In [ ]:

data = pd.concat([disprot_df, uniprot_df], ignore_index=True)
data = data.drop_duplicates(subset="sequence").reset_index(drop=True)

print("Combined dataset size:", len(data))
print(data["label"].value_counts())

data.head()

## 5. Save the input data

This creates a CSV file that students can reuse without downloading again.

In [ ]:

data.to_csv("globular_vs_disordered_input_data.csv", index=False)
print("Saved: globular_vs_disordered_input_data.csv")

## 6. Define simple sequence features

We deliberately use **hand-crafted features** so students can understand them.

Features used:
- sequence length
- fraction hydrophobic residues
- fraction polar residues
- fraction positively charged residues
- fraction negatively charged residues
- fraction glycine
- fraction proline
- fraction aromatic residues
- fraction disorder-promoting residues (very rough)
- fraction order-promoting residues (very rough)
- amino-acid composition for the 20 standard amino acids

In [ ]:

AA20 = list("ACDEFGHIKLMNPQRSTVWY")
HYDROPHOBIC = set("AVLIMFWY")
POLAR = set("STNQCY")
POSITIVE = set("KRH")
NEGATIVE = set("DE")
AROMATIC = set("FWYH")
DISORDER_PRONE = set("PGSQEKAR")
ORDER_PRONE = set("WFYILVNC")


def aa_fraction(seq, aa_set):
    if len(seq) == 0:
        return 0.0
    return sum(1 for aa in seq if aa in aa_set) / len(seq)


def sequence_to_features(seq):
    seq = seq.upper()
    feats = {
        "length": len(seq),
        "frac_hydrophobic": aa_fraction(seq, HYDROPHOBIC),
        "frac_polar": aa_fraction(seq, POLAR),
        "frac_positive": aa_fraction(seq, POSITIVE),
        "frac_negative": aa_fraction(seq, NEGATIVE),
        "frac_gly": aa_fraction(seq, {"G"}),
        "frac_pro": aa_fraction(seq, {"P"}),
        "frac_aromatic": aa_fraction(seq, AROMATIC),
        "frac_disorder_prone": aa_fraction(seq, DISORDER_PRONE),
        "frac_order_prone": aa_fraction(seq, ORDER_PRONE),
    }

    for aa in AA20:
        feats[f"aa_{aa}"] = aa_fraction(seq, {aa})

    return feats

## 7. Convert sequences into a feature matrix

In [ ]:

feature_df = pd.DataFrame([sequence_to_features(seq) for seq in data["sequence"]])
X = feature_df.copy()
y = data["label"].copy()

print("Feature matrix shape:", X.shape)
X.head()

## 8. Train/test split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

## 9. Train a basic logistic regression classifier

In [ ]:

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred, target_names=["disordered-like", "globular-like"]))

## 10. Confusion matrix

In [ ]:

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["disordered-like", "globular-like"])
disp.plot()
plt.show()

## 11. ROC curve

In [ ]:

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve")
plt.legend()
plt.show()

## 12. Inspect feature importance

For logistic regression, the sign and magnitude of the coefficients are useful:

- positive coefficient → pushes prediction toward **globular-like**
- negative coefficient → pushes prediction toward **disordered-like**

In [ ]:

clf = model.named_steps["clf"]
coefs = pd.Series(clf.coef_[0], index=X.columns).sort_values()

print("Most negative coefficients (more disordered-like):")
display(coefs.head(10).to_frame("coefficient"))

print("Most positive coefficients (more globular-like):")
display(coefs.tail(10).sort_values(ascending=False).to_frame("coefficient"))

## 13. Predict on a few example sequences

In [ ]:

example_sequences = [
    "MSEQQARKKDDGGLGGLGQGQGQGQGQGQGQ",
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDL",
    "GSSGSSGSSGSSGSSGSSGSSGSSGSS",
]

example_X = pd.DataFrame([sequence_to_features(seq) for seq in example_sequences])
example_prob = model.predict_proba(example_X)[:, 1]

results = pd.DataFrame({
    "sequence": example_sequences,
    "prob_globular_like": example_prob,
    "predicted_class": np.where(example_prob >= 0.5, "globular-like", "disordered-like")
})

results

## 14. Extension exercises for students

The cells below provide ready-to-run versions of the following extension ideas:

1. Replace logistic regression with a random forest.
2. Add or remove features and see how performance changes.
3. Change the UniProt query and discuss whether the task becomes easier or harder.
4. Plot sequence length distributions for the two classes.
5. Try balancing the dataset differently.

## 15. Optional extension cells

These cells turn the ideas above into small experiments you can run directly.

They are designed so you can compare one change at a time while keeping the rest of the workflow the same.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_SEED,
    class_weight="balanced",
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random forest accuracy:", round(accuracy_score(y_test, rf_pred), 3))
print()
print(classification_report(y_test, rf_pred, target_names=["disordered-like", "globular-like"]))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_pred,
    display_labels=["disordered-like", "globular-like"]
)
plt.title("Random forest confusion matrix")
plt.show()

### 15a. Add or remove features and compare performance

In [ ]:
# Compare different feature sets
# Create a few derived features so all subsets exist

X_compare = X.copy()
X_compare["frac_charged"] = X_compare["frac_positive"] + X_compare["frac_negative"]
X_compare["frac_gly_pro"] = X_compare["frac_gly"] + X_compare["frac_pro"]
X_compare["frac_disorder_promoting"] = X_compare["frac_disorder_prone"]

feature_sets = {
    "all_features": list(X_compare.columns),
    "composition_only": [c for c in X_compare.columns if c.startswith("frac_")],
    "no_length": [c for c in X_compare.columns if c != "length"],
    "compact_set": ["length", "frac_hydrophobic", "frac_charged", "frac_gly_pro", "frac_disorder_promoting"],
}

comparison_rows = []

for name, cols in feature_sets.items():
    X_sub = X_compare[cols]
    X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(
        X_sub,
        y,
        test_size=0.25,
        random_state=RANDOM_SEED,
        stratify=y,
    )

    sub_model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))
    ])

    sub_model.fit(X_train_sub, y_train_sub)
    y_pred_sub = sub_model.predict(X_test_sub)

    comparison_rows.append({
        "feature_set": name,
        "n_features": len(cols),
        "accuracy": accuracy_score(y_test_sub, y_pred_sub),
    })

pd.DataFrame(comparison_rows).sort_values("accuracy", ascending=False)

### 15b. Change the UniProt query

Try a different query below, regenerate the UniProt class, rebuild the combined dataset, and retrain the model.

Questions to discuss:
- Does the task become easier if the UniProt set contains very enzyme-like, well-folded proteins?
- Does it become harder if the query is broader and includes more diverse proteins?
- Does the sequence length distribution change?

In [ ]:
import requests
import pandas as pd
from io import StringIO

def fetch_uniprot_sequences(query, max_records=500, min_len=50, max_len=500):
    """
    Fetch protein sequences from UniProt as a pandas DataFrame.
    """

    full_query = (
        f"({query}) AND fragment:false "
        f"AND length:[{min_len} TO {max_len}]"
    )

    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": full_query,
        "format": "tsv",
        "fields": "accession,protein_name,length,sequence",
        "size": max_records,
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text), sep="\t")
    df = df.dropna(subset=["Sequence"]).copy()
    return df

In [ ]:
# Self-contained cell: alternative UniProt query experiment + data retrieval + model training

import requests
import pandas as pd
from io import StringIO

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# Parameters
# -----------------------------
N_PER_CLASS = 500
MIN_LEN = 50
MAX_LEN = 400
RANDOM_SEED = 42

# Change this to test whether the task becomes easier or harder
uniprot_query = "reviewed:true"

# -----------------------------
# Helper functions
# -----------------------------
def fetch_uniprot_sequences(query, max_records=500, min_len=50, max_len=400):
    """
    Fetch protein sequences from UniProt and return a DataFrame.
    Expected output columns include: accession, protein_name, Length, Sequence
    """
    full_query = f"({query}) AND fragment:false AND length:[{min_len} TO {max_len}]"

    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": full_query,
        "format": "tsv",
        "fields": "accession,protein_name,length,sequence",
        "size": max_records,
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text), sep="\t")

    # UniProt usually returns 'Sequence' and 'Length'
    if "Sequence" not in df.columns:
        raise ValueError(f"Expected a 'Sequence' column from UniProt, got: {list(df.columns)}")

    df = df.dropna(subset=["Sequence"]).copy()
    return df


def make_labelled_dataset(disprot_df, uniprot_df):
    """
    Combine DisProt sequences (label 0) and UniProt sequences (label 1)
    into one labelled dataset.
    """

    if "sequence" not in disprot_df.columns:
        raise ValueError("disprot_df must contain a 'sequence' column")

    if "Sequence" not in uniprot_df.columns:
        raise ValueError("uniprot_df must contain a 'Sequence' column")

    dis_df = disprot_df.copy()
    dis_df["label"] = 0
    dis_df["class_name"] = "disordered"
    dis_df = dis_df.rename(columns={"sequence": "sequence"})
    dis_df = dis_df[["sequence", "label", "class_name"]]

    uni_df = uniprot_df.copy()
    uni_df["label"] = 1
    uni_df["class_name"] = "globular"
    uni_df = uni_df.rename(columns={"Sequence": "sequence"})
    uni_df = uni_df[["sequence", "label", "class_name"]]

    combined = pd.concat([dis_df, uni_df], ignore_index=True)
    return combined


# -----------------------------
# Safety checks
# -----------------------------
if "disprot_df" not in globals():
    raise NameError("disprot_df is not defined. Run the earlier DisProt retrieval cell first.")

if "sequence_to_features" not in globals():
    raise NameError("sequence_to_features is not defined. Run the earlier feature-extraction cell first.")

# -----------------------------
# Fetch alternative UniProt set
# -----------------------------
print(f"Fetching UniProt sequences with query: {uniprot_query}")
uniprot_alt = fetch_uniprot_sequences(
    query=uniprot_query,
    max_records=N_PER_CLASS,
    min_len=MIN_LEN,
    max_len=MAX_LEN,
)

print(f"Retrieved {len(uniprot_alt)} UniProt sequences")

# -----------------------------
# Build labelled dataset
# -----------------------------
alt_data = make_labelled_dataset(disprot_df, uniprot_alt)

# Optional: rebalance to the same number per class
min_class_size = alt_data["class_name"].value_counts().min()
alt_data = (
    alt_data.groupby("class_name", group_keys=False)
    .apply(lambda d: d.sample(min_class_size, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

print("\nClass counts after balancing:")
print(alt_data["class_name"].value_counts())

# -----------------------------
# Convert sequences to features
# -----------------------------
alt_feature_df = pd.DataFrame(
    [sequence_to_features(seq) for seq in alt_data["sequence"]]
)

X_alt = alt_feature_df.copy()
y_alt = alt_data["label"].copy()

# -----------------------------
# Train/test split
# -----------------------------
X_train_alt, X_test_alt, y_train_alt, y_test_alt = train_test_split(
    X_alt,
    y_alt,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_alt,
)

# -----------------------------
# Random Forest model
# -----------------------------
alt_model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_SEED,
    class_weight="balanced",
    n_jobs=-1,
)

alt_model.fit(X_train_alt, y_train_alt)
alt_pred = alt_model.predict(X_test_alt)

# -----------------------------
# Results
# -----------------------------
print("\nUniProt query used:", uniprot_query)
print("Accuracy:", round(accuracy_score(y_test_alt, alt_pred), 3))
print()
print(classification_report(y_test_alt, alt_pred))

### 15c. Plot sequence length distributions for the two classes

In [ ]:
# Plot sequence length distributions for the two classes

plot_df = data.copy()
plot_df["length"] = plot_df["sequence"].str.len()

# Recreate class_name if it is missing
if "class_name" not in plot_df.columns:
    if "label" in plot_df.columns:
        plot_df["class_name"] = plot_df["label"].map({0: "disordered", 1: "globular"})
    else:
        raise ValueError("data must contain either 'class_name' or 'label'.")

plt.figure(figsize=(7, 5))
for class_name in plot_df["class_name"].unique():
    lengths = plot_df.loc[plot_df["class_name"] == class_name, "length"]
    plt.hist(lengths, bins=20, alpha=0.6, label=class_name)

plt.xlabel("Sequence length")
plt.ylabel("Count")
plt.title("Sequence length distributions by class")
plt.legend()
plt.show()

plot_df.groupby("class_name")["length"].describe()

### 15d. Try balancing the dataset differently

This cell rebuilds the dataset with a different number of sequences per class.

Try changing `N_PER_CLASS_ALT` and compare the results with the original run.

In [ ]:
N_PER_CLASS_ALT = 150

disprot_alt = disprot_df.sample(
    n=min(N_PER_CLASS_ALT, len(disprot_df)),
    random_state=RANDOM_SEED
).reset_index(drop=True)

uniprot_balanced_alt = uniprot_df.sample(
    n=min(N_PER_CLASS_ALT, len(uniprot_df)),
    random_state=RANDOM_SEED
).reset_index(drop=True)

# Standardize UniProt sequence column
if "Sequence" in uniprot_balanced_alt.columns and "sequence" not in uniprot_balanced_alt.columns:
    uniprot_balanced_alt = uniprot_balanced_alt.rename(columns={"Sequence": "sequence"})

if "sequence" not in disprot_alt.columns:
    raise ValueError("disprot_alt must contain a 'sequence' column")

if "sequence" not in uniprot_balanced_alt.columns:
    raise ValueError("uniprot_balanced_alt must contain a 'sequence' column")

# Build labelled dataset directly
disprot_part = disprot_alt[["sequence"]].copy()
disprot_part["label"] = 0
disprot_part["class_name"] = "disordered"

uniprot_part = uniprot_balanced_alt[["sequence"]].copy()
uniprot_part["label"] = 1
uniprot_part["class_name"] = "globular"

balanced_alt_data = pd.concat(
    [disprot_part, uniprot_part],
    ignore_index=True
)

balanced_alt_features = pd.DataFrame(
    [sequence_to_features(seq) for seq in balanced_alt_data["sequence"]]
)

X_bal = balanced_alt_features.copy()
y_bal = balanced_alt_data["label"].copy()

X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_bal,
    y_bal,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_bal,
)

balanced_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))
])

balanced_model.fit(X_train_bal, y_train_bal)
balanced_pred = balanced_model.predict(X_test_bal)

print("Balanced dataset size:", len(balanced_alt_data))
print(balanced_alt_data["class_name"].value_counts())
print()
print("Accuracy:", round(accuracy_score(y_test_bal, balanced_pred), 3))